# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Maryam/Aleeza-ML-Internship-WEEK1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)
print("Token starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)

Token exists: True
Token starts correctly: True


In [2]:
!pip -q install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 10.6 MB/s eta 0:00:00


In [3]:
from huggingface_hub import whoami

info = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("User:", info["name"])

Hugging Face authentication successful.
User: Aleeza50


In [5]:
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print(HF_DATASET)

hf://datasets/FlyRank/internship-warehouse


In [11]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)
print("Token starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
import duckdb

con = duckdb.connect()

print("DuckDB connected")
try:
    con.execute("DROP SECRET IF EXISTS hf_secret")
except:
    pass

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection configured")
test = con.sql(f"""
SELECT *
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""")

test.show()
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS gsc_impressions_present,
    COUNT(gsc_clicks) AS gsc_clicks_present,
    COUNT(gsc_avg_position) AS gsc_avg_position_present,
    COUNT(ga4_pageviews) AS ga4_pageviews_present,
    COUNT(ga4_sessions) AS ga4_sessions_present,
    COUNT(ga4_users) AS ga4_users_present,
    COUNT(ga4_engaged_sessions) AS ga4_engaged_sessions_present,
    COUNT(ga4_total_engagement_sec) AS ga4_engagement_present
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

availability_check.show()

Token exists: True
Token starts correctly: True
DuckDB connected
Hugging Face connection configured
┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────────┬────────────────────┬──────────────────────────┬───────────────────────┬──────────────────────┬───────────────────┬──────────────────────────────┬────────────────────────┐
│ total_rows │ gsc_impressions_present │ gsc_clicks_present │ gsc_avg_position_present │ ga4_pageviews_present │ ga4_sessions_present │ ga4_users_present │ ga4_engaged_sessions_present │ ga4_engagement_present │
│   int64    │          int64          │       int64        │          int64           │         int64         │        int64         │       int64       │            int64             │         int64          │
├────────────┼─────────────────────────┼────────────────────┼──────────────────────────┼───────────────────────┼──────────────────────┼───────────────────┼──────────────────────────────┼────────────────────────┤
│    9841378 │                 9841378 │            9841378 │                  3611061 │               6822637 │              6822637 │           682263

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 10000
""").df()

feature_df.head(10)

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


In [13]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

print("Feature frame shape:", feature_df.shape)

print("\nMissing values:")
print(feature_df[feature_columns].isna().sum())

Feature frame shape: (10000, 8)

Missing values:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position         1629
ga4_pageviews           10000
ga4_engaged_sessions    10000
dtype: int64


### Five features and when they are available

1. **gsc_impressions** — Knowable at the decision moment because Search Console impressions are observed before deciding whether a content item needs a refresh.

2. **gsc_clicks** — Knowable at the decision moment because Search Console clicks are observed search-performance signals available before the refresh decision.

3. **gsc_avg_position** — Knowable at the decision moment because average search position is an observed ranking signal. It is only available when GSC data is available.

4. **ga4_pageviews** — Knowable at the decision moment because pageviews are observed website-analytics activity before the refresh decision. It is available when GA4 data is available.

5. **ga4_engaged_sessions** — Knowable at the decision moment because engaged sessions are observed user-engagement activity before the refresh decision. It is available when GA4 data is available.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature frame shape:", feature_df.shape)

print("\nMissing values:")
print(feature_df[feature_columns].isna().sum())

Feature frame shape: (10000, 8)

Missing values:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position         1629
ga4_pageviews           10000
ga4_engaged_sessions    10000
dtype: int64


In [15]:
march_path = f"{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet"
april_path = f"{HF_DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet"

march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{march_path}')
""").df()

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions AS april_impressions
FROM read_parquet('{april_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
leak_df = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leak_df["future_decline"] = (
    leak_df["april_impressions"] < leak_df["gsc_impressions"]
).astype(int)

print("Rows:", len(leak_df))
print("Future decline rate:", leak_df["future_decline"].mean())

In [ ]:
leak_df["LEAKED_FEATURE"] = leak_df["future_decline"]

### Leakage experiment

I deliberately added `future_decline` as a feature. This feature is derived from April performance, which occurs after the March decision moment. Therefore, it would not have been available when the refresh decision was made.

This is data leakage because the model is being given information derived from the future outcome.

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)
print("Token starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
import duckdb

con = duckdb.connect()

print("DuckDB connected")
try:
    con.execute("DROP SECRET IF EXISTS hf_secret")
except:
    pass
march_path = f"{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet"
april_path = f"{HF_DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet"

march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{march_path}')
""").df()

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions AS april_impressions
FROM read_parquet('{april_path}')
""").df()

print("March rows:", len(march))
print("April rows:", len(april))



Token exists: True
Token starts correctly: True
DuckDB connected


In [ ]:
leak_df = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leak_df["future_decline"] = (
    leak_df["april_impressions"] < leak_df["gsc_impressions"]
).astype(int)

print("Rows after matching:", len(leak_df))
print("Future decline rate:", round(leak_df["future_decline"].mean(), 3))

In [ ]:
leak_df["LEAKED_FEATURE"] = leak_df["future_decline"]

print(
    "Leaked feature equals label:",
    (leak_df["LEAKED_FEATURE"] == leak_df["future_decline"]).all()
)

### Leakage trap

I deliberately added `LEAKED_FEATURE`, which is exactly the future decline label.

This is leakage because the feature uses April information, while the decision is being made using March information. April performance would not be known at the March decision moment.

A model using this feature can appear unrealistically accurate because it has effectively been given the answer.

The leaked feature must therefore be removed before any honest evaluation.

In [ ]:
leak_df = leak_df.drop(columns=["LEAKED_FEATURE"])

print("LEAKED_FEATURE removed.")
print("Remaining columns:")
print(leak_df.columns.tolist())

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.